# Exportación modelos

Valida artefactos runtime y prepara copias versionadas cuando `RUN` sea `True`.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import shutil
import joblib
import pandas as pd

ROOT = Path.cwd()
if (ROOT / 'Suplematch-Backend').exists():
    ROOT = ROOT / 'Suplematch-Backend'
while ROOT.name != 'Suplematch-Backend' and ROOT.parent != ROOT:
    ROOT = ROOT.parent

runtime = ROOT / 'models/runtime'
assert runtime.exists()
ROOT

## Artefactos

In [ ]:
artifacts = sorted(runtime.glob('*'))
pd.DataFrame([{'file': path.name, 'size': path.stat().st_size, 'suffix': path.suffix} for path in artifacts if path.is_file()])

## Carga

In [ ]:
load_results = []
for path in runtime.glob('*.pkl'):
    try:
        obj = joblib.load(path)
        load_results.append({'file': path.name, 'loaded': True, 'type': type(obj).__name__})
    except Exception as exc:
        load_results.append({'file': path.name, 'loaded': False, 'type': type(exc).__name__})
pd.DataFrame(load_results)

## Copia versionada

In [ ]:
RUN = False
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
export_dir = ROOT / 'models/backups' / stamp
copied = []
if RUN:
    export_dir.mkdir(parents=True, exist_ok=True)
    for path in runtime.glob('*'):
        if path.is_file():
            target = export_dir / path.name
            shutil.copy2(path, target)
            copied.append(str(target.relative_to(ROOT)))
{'export_dir': str(export_dir.relative_to(ROOT)), 'executed': RUN, 'copied': copied}